# RNN实现

## 读取词表和数据集

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader
from text_preprocessing import get_vocab_corpus_from_timemachine, Vocabulary

class TextDataset(Dataset):
    def __init__(self, corpus: list[int], seq_length: int):
        self.corpus = corpus
        self.seq_length = seq_length

    def __len__(self):
        return len(self.corpus) - self.seq_length

    def __getitem__(self, idx):
        # 输入序列和右移一位得到的目标序列
        return (torch.tensor(self.corpus[idx: idx + self.seq_length]),
                torch.tensor(self.corpus[idx + 1: idx + self.seq_length + 1]))

def timemachine_data_loader(batch_size: int,
                            seq_length: int,
                            shuffle:bool = False,
                            max_token_num = 10000) -> tuple[DataLoader, Vocabulary]:
    vocab, corpus = get_vocab_corpus_from_timemachine(token_type= 'char')
    data_iter = DataLoader(TextDataset(corpus, seq_length),batch_size, shuffle, drop_last=True)
    return data_iter, vocab

## 模型参数初始化

根据 RNN 的模型

$$
H_t = \sigma ( X_t W_{xh} + H_{t-1}W_{hh} + b_h )
$$
$$
O_t = H_t W_{hq} + b_q
$$

其中 $X$ 为 $n\times d$ 的小批量，

可学习参数包括

- 输入层到隐藏层权重 `w_input2hidden`
- 隐状态到隐藏层的权重 `w_hidden2hidden`
- 隐藏层的偏置 `b_hidden`
- 隐藏层到输出的权重 `w_hidden2output`
- 隐藏层到输出的权重 `b_output`

所有权重由均值为 0 ，方差为 0.01 的标准正态分布初始化，偏置均初始化为 0 。

In [12]:
from torch import Tensor, nn


def init_params(vocab_size: int,
                hidden_num: int,
                device: torch.device | str) -> tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
    w_input2hidden = torch.randn((vocab_size, hidden_num), device=device) * 0.01
    w_hidden2hidden = torch.randn((hidden_num, hidden_num), device=device) * 0.01
    w_hidden2output = torch.randn((hidden_num, vocab_size), device=device) * 0.01
    b_hidden = torch.zeros(hidden_num, device=device)
    b_output = torch.zeros(vocab_size, device=device)

    return (w_input2hidden.requires_grad_(),
            w_hidden2hidden.requires_grad_(),
            b_hidden.requires_grad_(),
            w_hidden2output.requires_grad_(),
            b_output.requires_grad_())

## 隐状态初始化


In [13]:
from typing import Tuple, Iterable


def init_hidden_states(batch_size: int,
                       hidden_num: int,
                       device: torch.device | str) -> Tuple[Tensor, ...]:
    return torch.zeros((batch_size, hidden_num), device=device),

## 单步隐状态计算

使用 $\tanh$ 函数激活。在模型训练初期，均匀分布数据的$\tanh$的均值为 0，避免了输出的偏移，保证了梯度的顺利传播。

In [14]:
def rnn_step(inputs: Tensor,
             states: Tuple[Tensor,...],
             params: Tuple[Tensor,...]) -> Tuple[Tensor, tuple]:
    w_input2hidden, w_hidden2hidden, b_hidden, w_hidden2output, b_output = params
    state, = states
    outputs_temp = []

    for step in inputs:
        state = torch.tanh(
            step @ w_input2hidden + b_hidden + state @ w_hidden2hidden
        )
        output_layer = state @ w_hidden2output + b_output
        outputs_temp.append(output_layer)

    outputs = torch.cat(outputs_temp, dim=0)
    out_states = (state, )
    return outputs, out_states

## 使用类封装

In [15]:
from typing import Protocol
import torch.nn.functional as F

class RNNScratch:
    def __init__(self, vocab_size: int, hidden_num: int, device: torch.device | str):
        self.__vocab_size = vocab_size
        self.__hidden_num = hidden_num
        self.params = self.__init_params(device)

    def __init_params(self,
                      device: torch.device | str) -> tuple[Tensor, Tensor, Tensor, Tensor, Tensor]:
        w_input2hidden = torch.randn((self.__vocab_size, self.__hidden_num), device=device) * 0.01
        w_hidden2hidden = torch.randn((self.__hidden_num, self.__hidden_num), device=device) * 0.01
        w_hidden2output = torch.randn((self.__hidden_num, self.__vocab_size), device=device) * 0.01
        b_hidden = torch.zeros(self.__hidden_num, device=device)
        b_output = torch.zeros(self.__vocab_size, device=device)

        return (w_input2hidden.requires_grad_(),
                w_hidden2hidden.requires_grad_(),
                b_hidden.requires_grad_(),
                w_hidden2output.requires_grad_(),
                b_output.requires_grad_())

    def init_hidden_states(self,
                           batch_size: int,
                           device: torch.device | str) -> Tuple[Tensor, ...]:
        return torch.zeros((batch_size, self.__hidden_num), device=device),

    @staticmethod
    def __rnn_step(inputs: Tensor,
                 states: Tuple[Tensor,...],
                 params: Tuple[Tensor,...]) -> Tuple[Tensor, tuple]:
        w_input2hidden, w_hidden2hidden, b_hidden, w_hidden2output, b_output = params
        state, = states
        outputs_temp = []

        for step in inputs:
            state = torch.tanh(
                step @ w_input2hidden + b_hidden + state @ w_hidden2hidden
            )
            output_layer = state @ w_hidden2output + b_output
            outputs_temp.append(output_layer)

        outputs = torch.cat(outputs_temp, dim=0)
        out_states = (state, )
        return outputs, out_states

    def __call__(self, inputs, states) -> Tuple[Tensor, tuple]:
        inputs = F.one_hot(inputs.T, self.__vocab_size).type(torch.float32)
        return self.__rnn_step(inputs, states= states, params= self.params)

## 定义预测函数

In [16]:
def forecast_chars(prefix: str,
                   num: int,
                   model: RNNScratch,
                   vocab: Vocabulary,
                   device: torch.device | str) -> str:
    states = model.init_hidden_states(batch_size=1, device=device)
    outputs = [vocab.get_index(prefix[0])]

    for y in prefix[1:]: # 预热
        _, states = model(torch.tensor(outputs[-1:], device=device).unsqueeze(0), states)
        outputs.append(vocab.get_index(y))

    for _ in range(num): # 预测
        y, states = model(torch.tensor(outputs[-1:], device=device).unsqueeze(0), states)
        outputs.append(torch.argmax(y, dim=1).item())

    return ''.join(vocab.decode(outputs))


指定前缀开始预测

In [17]:
if __name__ == '__main__':

    BATCH_SIZE = 32
    SEQ_LENGTH = 35
    HIDDEN_NUM = 512

    data_iter, vocab = timemachine_data_loader(BATCH_SIZE, SEQ_LENGTH)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    rnn = RNNScratch(vocab_size=len(vocab), hidden_num=HIDDEN_NUM, device=device)
    init_states = rnn.init_hidden_states(BATCH_SIZE, device)

    prefix = 'time traveller '
    step_num = 10
    prediction = forecast_chars(prefix, step_num, rnn, vocab, device)
    print(f'前缀{prefix!r}经 {step_num} 次预测后为：{prediction!r}')

前缀'time traveller '经 10 次预测后为：'time traveller srufopgvll'


## 模型训练与评估

为防止梯度爆炸或消失，可使用梯度裁剪
当梯度的范数 $||\nabla||$ 超过特定值 $\theta$ 时，缩小 $\frac{\theta}{||\nabla||}$ 倍
$$
\nabla = min(1, \frac{\theta}{||\nabla||}) \nabla
$$

In [18]:
from typing import Iterable


def clip_gradients(model: nn.Module | RNNScratch, max_norm: float):
    params: Iterable[Tensor]
    params = [p for p in model.parameters() if p.requires_grad] if isinstance(model, nn.Module) else model.params

    grad_l2_norm = torch.norm(torch.cat([p.grad.flatten() for p in params]), 2)
    if grad_l2_norm > max_norm:
        for p in params:
            p.grad.mul_(max_norm / grad_l2_norm)

In [19]:
from time import time
from typing import Callable, Optional
from math import exp
from torch import optim

def train_one_epoch(
        net: nn.Module | RNNScratch,
        date_iter: Iterable[tuple[[Tensor,Tensor]]],
        loss_fn: nn.Module,
        updater: optim.Optimizer | Callable,
        device: torch.device,
        shuffle: bool = False
) -> Tuple[float, float]:
    """
    在一个迭代周期内训练模型

    :param net: RNN 实例
    :param data_iter: 数据集加载器
    :param loss_fn: 损失函数
    :param updater: 优化器或参数更新函数
    :param device: 计算设备
    :param shuffle: 是否随机采样
    :return: 困惑度, 训练速度
    """
    states: Optional[Tuple[Tensor, ...]] = None
    total_tokens = 0
    total_loss = 0.0
    start_time = time()

    for features, labels in data_iter:
        features = features.to(device)
        labels = labels.T.flatten().to(device)
        if shuffle or states is None:
            states = net.init_hidden_states(features.shape[0], device=device)
        else:
            states = tuple(s.detach() for s in states)

        output, states = net(features, states)
        loss = loss_fn(output, labels)

        if isinstance(updater, optim.Optimizer):
            updater.zero_grad()
            loss.backward()
            clip_gradients(net, max_norm=1)
            updater.step()
        else:
            loss.backward()
            clip_gradients(net, max_norm=1)
            updater(features.shape[0])

        batch_tokens = labels.numel()
        total_loss += loss.item() * batch_tokens
        total_tokens += batch_tokens

    perplexity = exp(total_loss / total_tokens)
    tokens_pre_sec = total_tokens / (time() - start_time)
    return perplexity, tokens_pre_sec

## 使用随机采样

In [ ]:
if __name__ == '__main__':

    BATCH_SIZE = 32
    SEQ_LENGTH = 35
    HIDDEN_NUM = 512
    EPOCHS_NUM = 1
    LEARNING_RATE = 0.7
    IS_SHUFFLE = True
    FORCAST_INTERVAL = 10
    PREFIX_STRING = 'time traveller'

    data_iter, vocab = timemachine_data_loader(BATCH_SIZE, SEQ_LENGTH, IS_SHUFFLE)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    rnn = RNNScratch(vocab_size=len(vocab), hidden_num=HIDDEN_NUM, device=device)
    optimizer = optim.SGD(rnn.params, lr=LEARNING_RATE)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(EPOCHS_NUM):
        ppl, speed = train_one_epoch(rnn, data_iter, loss_fn, optimizer, device, IS_SHUFFLE)
        print(f'第 {epoch + 1:02} 轮：困惑度为 {ppl:04.1f}，速度为 {speed:.1f} (tokens/sec)')

        if (epoch + 1) % FORCAST_INTERVAL == 0:
            with torch.no_grad():  # 评估模式
                prediction = forecast_chars(PREFIX_STRING, 50, rnn, vocab, device)
                print(f'预测结果：{prediction!r}')